# Finalisasi preprocessing tanpa mengulang YOLO
Notebook ini membaca output preprocessing yang sudah selesai, membuang *window* yang benar-benar rendah gerakan, menyelaraskan seluruh array, membuat ulang indeks split, memeriksa kebocoran video, dan menyimpan dataset final.

In [1]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd

MIN_MOTION_SCORE = 0.015
MIN_ACTIVE_TRANSITION_RATIO = 0.10
MAX_REJECT_RATIO_PER_CLASS = 0.35

INPUT_ROOT = Path('/kaggle/input/datasets/wafabila/preprocessing-not-filtered')
INPUT_ARRAY_DIR = INPUT_ROOT / 'arrays'
OUTPUT_DIR = Path('/kaggle/working/preprocessed_final_low_motion_filtered')
ARRAY_DIR = OUTPUT_DIR / 'arrays'
REPORT_DIR = OUTPUT_DIR / 'reports'
ARRAY_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Input eksplisit dari dataset Save Version preprocessing tanpa filter.
if not INPUT_ROOT.exists():
    raise FileNotFoundError(f'Dataset input tidak ditemukan: {INPUT_ROOT}')
if not INPUT_ARRAY_DIR.exists():
    raise FileNotFoundError(f'Folder arrays tidak ditemukan: {INPUT_ARRAY_DIR}')

print('Dataset:', INPUT_ROOT)
print('Input :', INPUT_ARRAY_DIR)
print('Output:', OUTPUT_DIR)

Dataset: /kaggle/input/datasets/wafabila/preprocessing-not-filtered
Input : /kaggle/input/datasets/wafabila/preprocessing-not-filtered/arrays
Output: /kaggle/working/preprocessed_final_low_motion_filtered


In [2]:
# Semua file berikut mempunyai baris yang sejajar dengan X_sequences.
aligned_names = [
    'X_sequences.npy', 'y_labels.npy', 'valid_masks.npy',
    'source_video_ids.npy', 'sequence_ids.npy', 'splits.npy',
    'start_frames.npy', 'end_frames.npy',
    'valid_frame_counts.npy', 'missing_ratios.npy',
    'motion_scores.npy', 'active_transition_ratios.npy',
]

arrays = {}
for name in aligned_names:
    path = INPUT_ARRAY_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'File wajib tidak ditemukan: {path}')
    arrays[name] = np.load(path, allow_pickle=False)

n = len(arrays['X_sequences.npy'])
for name, array in arrays.items():
    if len(array) != n:
        raise RuntimeError(f'Panjang {name}={len(array)} tidak sama dengan X={n}')

X = arrays['X_sequences.npy']
if X.ndim != 3 or X.shape[1:] != (30, 51) or X.dtype != np.float32:
    raise RuntimeError(f'X tidak sesuai: shape={X.shape}, dtype={X.dtype}')

motion = arrays['motion_scores.npy'].astype(np.float32)
active = arrays['active_transition_ratios.npy'].astype(np.float32)
labels = arrays['y_labels.npy']

low_motion = (motion < MIN_MOTION_SCORE) & (active < MIN_ACTIVE_TRANSITION_RATIO)
keep = ~low_motion

rows = []
for class_id in sorted(np.unique(labels).tolist()):
    class_mask = labels == class_id
    total = int(class_mask.sum())
    rejected = int((class_mask & low_motion).sum())
    rows.append({
        'class_id': int(class_id),
        'before': total,
        'rejected': rejected,
        'after': total - rejected,
        'rejected_ratio': rejected / max(total, 1),
    })
audit = pd.DataFrame(rows)
display(audit)

if (audit['after'] == 0).any():
    raise RuntimeError('Filter menghabiskan salah satu kelas. Dataset tidak disimpan.')
if (audit['rejected_ratio'] > MAX_REJECT_RATIO_PER_CLASS).any():
    bad = audit[audit['rejected_ratio'] > MAX_REJECT_RATIO_PER_CLASS]
    raise RuntimeError(f'Lebih dari 35% salah satu kelas akan dibuang:\n{bad}')

print('Sebelum :', n)
print('Dibuang :', int(low_motion.sum()))
print('Disimpan:', int(keep.sum()))

,class_id,before,rejected,after,rejected_ratio
0,0,3043,0,3043,0.000000
1,1,3175,0,3175,0.000000
2,2,2979,0,2979,0.000000
3,3,3023,0,3023,0.000000
4,4,3163,0,3163,0.000000
5,5,2912,0,2912,0.000000
6,6,3057,0,3057,0.000000
7,7,3093,1,3092,0.000323
8,8,2997,0,2997,0.000000


Sebelum : 27442
Dibuang : 1
Disimpan: 27441


In [3]:
# Filter seluruh array secara serentak agar tidak terjadi salah pasangan data-label.
filtered = {name: array[keep] for name, array in arrays.items()}
split_array = filtered['splits.npy'].astype(str)
train_indices = np.where(split_array == 'train')[0].astype(np.int64)
validation_indices = np.where(split_array == 'validation')[0].astype(np.int64)
test_indices = np.where(split_array == 'test')[0].astype(np.int64)

# Pemeriksaan split video: satu video tidak boleh muncul pada dua subset.
source_ids = filtered['source_video_ids.npy'].astype(str)
video_sets = {name: set(source_ids[split_array == name])
              for name in ['train', 'validation', 'test']}
overlap = {
    'train_validation': sorted(video_sets['train'] & video_sets['validation']),
    'train_test': sorted(video_sets['train'] & video_sets['test']),
    'validation_test': sorted(video_sets['validation'] & video_sets['test']),
}
if any(overlap.values()):
    raise RuntimeError(f'Ditemukan source-video leakage: {overlap}')

for name, array in filtered.items():
    np.save(ARRAY_DIR / name, array)
np.save(ARRAY_DIR / 'train_indices.npy', train_indices)
np.save(ARRAY_DIR / 'validation_indices.npy', validation_indices)
np.save(ARRAY_DIR / 'test_indices.npy', test_indices)

mapping_source = INPUT_ROOT / 'class_mapping.json'
if mapping_source.exists():
    shutil.copy2(mapping_source, OUTPUT_DIR / 'class_mapping.json')

audit.to_csv(REPORT_DIR / 'low_motion_filter_by_class.csv', index=False)
summary = {
    'operation': 'post_filter_low_motion_without_rerunning_yolo',
    'minimum_motion_score': MIN_MOTION_SCORE,
    'minimum_active_transition_ratio': MIN_ACTIVE_TRANSITION_RATIO,
    'filter_logic': 'motion_score < threshold AND active_transition_ratio < threshold',
    'total_sequences_before': int(n),
    'rejected_low_motion_sequences': int(low_motion.sum()),
    'total_sequences_after': int(keep.sum()),
    'X_shape': list(filtered['X_sequences.npy'].shape),
    'X_dtype': str(filtered['X_sequences.npy'].dtype),
    'split_counts': {
        'train': int(len(train_indices)),
        'validation': int(len(validation_indices)),
        'test': int(len(test_indices)),
    },
    'source_video_leakage': overlap,
}
with (OUTPUT_DIR / 'preprocessing_summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

archive = shutil.make_archive('/kaggle/working/preprocessed_final_low_motion_filtered', 'zip', OUTPUT_DIR)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('ZIP:', archive)

{
  "operation": "post_filter_low_motion_without_rerunning_yolo",
  "minimum_motion_score": 0.015,
  "minimum_active_transition_ratio": 0.1,
  "filter_logic": "motion_score < threshold AND active_transition_ratio < threshold",
  "total_sequences_before": 27442,
  "rejected_low_motion_sequences": 1,
  "total_sequences_after": 27441,
  "X_shape": [
    27441,
    30,
    51
  ],
  "X_dtype": "float32",
  "split_counts": {
    "train": 20629,
    "validation": 4034,
    "test": 2778
  },
  "source_video_leakage": {
    "train_validation": [],
    "train_test": [],
    "validation_test": []
  }
}
ZIP: /kaggle/working/preprocessed_final_low_motion_filtered.zip
